# Train model ViT + CNN according to paper : 
 

Points de vigilance 

Le papier reste assez vague sur les dimensions exactes (combien de num_stages, profondeur exacte du Transformer, valeur précise de reduction dans l'ECAM). J'ai pris des valeurs raisonnables — à toi de les recaler si tu veux reproduire les 86.8%.
L'opération "unfold/deconvolution" décrite dans le papier (section 2.1) est ambiguë dans le texte — j'ai interprété ça comme un cycle downsample→Transformer→upsample classique façon architecture en U, ce qui est l'interprétation la plus cohérente avec le schéma décrit, mais ce n'est pas garanti à 100% fidèle à leur implémentation (ils n'ont pas publié de code, j'ai vérifié — la section Data Availability ne mentionne que le dataset GTZAN, pas de repo).
Comparé à ton architecture CNN_AudioSpectralFeatureV2 (fusion spectrogramme + features librosa), ce papier reste mono-modal (que le Mel spectrogramme) — pas de fusion avec des features numériques. Le ECAM pourrait potentiellement remplacer/compléter ta branche CNN actuelle si tu veux tester une variante.

librairies & variables 

In [14]:
# Importation des bibliothèques #######
from calendar import EPOCH
import os, shutil
import pandas as pd
import plotly.express as px
import numpy as np
import torch
from torchinfo import summary
import torchvision.transforms.v2 as transforms
from torchvision import datasets
import torchaudio
import torchaudio.transforms as T

from PIL import Image
import requests
from io import BytesIO
import matplotlib.pyplot as plt
import plotly.graph_objects as go

import torch.nn as nn
import torch.optim as optim
import librosa
import seaborn
import boto3
from dotenv import load_dotenv
import mlflow
import mlflow.pytorch
from sklearn.metrics import ConfusionMatrixDisplay, f1_score
import datetime
from torchviz import make_dot
import argparse
import subprocess
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split
import sys
from mlflow.models import infer_signature
from sklearn.metrics import accuracy_score, f1_score



#######

####### VARIABLES ENVIRONNEMENT ET GENERALES #######
# Chargement des variables d'environnement
load_dotenv()
# Initialisation et création des répertoires s'ils n'existent pas #######
REP_KAGGLE = "https://www.kaggle.com/api/v1/datasets/download/andradaolteanu/gtzan-dataset-music-genre-classification"
ZIP_FILE = "gtzan-dataset-music-genre-classification.zip"
# Locaux de base
PATH_BASE = "gtzan-dataset-music-genre-classification"
PATH_DATA = PATH_BASE + "/Data"
# Local des images RGB de spectrogrammes entiers
PATH_IMAGE = PATH_BASE + "/Data/images_original"
# Local des sons à partir desquels les spectrogrammes seront/ont été générés
PATH_SOUND = PATH_BASE + "/Data/genres_original"
# Local des images grey des spectrogrammes harmoniques
PATH_HARMO = PATH_BASE + "/Data/img_harmo"
# Local des images grey des spectrogrammes percussifs
PATH_PERCU = PATH_BASE + "/Data/img_percu"
PATH_SPEC_VIT = PATH_BASE + "/Data/img_ViT"
# Le dataset nous facilitant les modules d'entrainements
PATH_DS = PATH_BASE + "/Data/features_30_sec.csv"
PATH_DS_SPLIT = PATH_BASE + "/Data/features_3_sec.csv"
#######

# Chargement des variables d'environnement
DATA_S3 = os.getenv("DATA_S3")
MLFLOW_URI = os.getenv("MLFLOW_TRACKING_URI")
#

# Variables d'entraînement ###
NUM_CLASSES = 10
####### FIN VARIABLES #######


####### FONCTIONS DE DIVERSES #######

# 0. computation of images 

In [3]:


"""Point d'attention pratique : torchaudio utilise un backend audio sous le capot 
(soundfile, sox, ou ffmpeg selon ta plateforme/version). 
Si tu rencontres une erreur de type RuntimeError: Couldn't find appropriate backend lors du chargement, 
vérifie que soundfile est installé (pip install soundfile --break-system-packages), 
c'est généralement le backend le plus simple et le plus portable sur Windows."""


def trim_silence(waveform: torch.Tensor, top_db: float = 60.0,
                  frame_length: int = 2048, hop_length: int = 512) -> torch.Tensor:
    """
    Équivalent torchaudio de librosa.effects.trim (seuil RMS en dB par frame).
    Pas d'équivalent natif strict dans torchaudio, donc réimplémentation manuelle.
    """
    abs_wav = waveform.abs()
    n_frames = 1 + (waveform.shape[-1] - frame_length) // hop_length
    if n_frames <= 0:
        return waveform

    frames = abs_wav.unfold(-1, frame_length, hop_length)   # (1, n_frames, frame_length)
    rms = frames.pow(2).mean(-1).sqrt().squeeze(0)           # (n_frames,)
    rms_db = 20 * torch.log10(rms + 1e-10)
    threshold = rms_db.max() - top_db

    above = (rms_db > threshold).nonzero(as_tuple=True)[0]
    if len(above) == 0:
        return waveform

    start_sample = above[0].item() * hop_length
    end_sample = min(above[-1].item() * hop_length + frame_length, waveform.shape[-1])
    return waveform[:, start_sample:end_sample]


def _to_db_normalized_uint8(mel_amplitude: torch.Tensor) -> np.ndarray:
    """
    Convertit un mel-spectrogramme d'AMPLITUDE (torch.Tensor) en image uint8 normalisée,
    avec flip vertical (axe fréquence).
    """
    # amplitude_to_db équivalent à librosa.amplitude_to_db(S, ref=np.max)
    db_transform = T.AmplitudeToDB(stype="amplitude", top_db=None)
    S_DB = db_transform(mel_amplitude)      # (1, n_mels, T), en dB, ref=1.0 par défaut

    # torchaudio n'a pas ref=np.max nativement -> réplique manuelle
    # amplitude_to_db(S, ref=np.max) == 20*log10(S) - 20*log10(S.max())
    S_DB = S_DB - S_DB.max()

    S_DB_np = S_DB.squeeze(0).numpy()       # (n_mels, T)
    S_DB_np = np.flipud(S_DB_np)            # même inversion verticale que l'original

    norm = (S_DB_np - S_DB_np.min()) / (S_DB_np.max() - S_DB_np.min() + 1e-8)
    return (norm * 255).astype(np.uint8)


def mel_spectrogram_amplitude(in_path: str, out_path: str, device: str = "cpu"):
    """
    Charge un fichier audio local, calcule son mel-spectrogramme en amplitude dB,
    et sauvegarde l'image résultante.
    """
    n_fft = 2048
    hop_length = 512
    n_mels = 128

    # --- Chargement (torchaudio gère directement un path local) ---
    waveform, sr = torchaudio.load(in_path)   # (channels, N)

    # Mono si stéréo (équivalent à librosa.load(mono=True), comportement par défaut)
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)

    # --- Trim silence ---
    waveform = trim_silence(waveform)

    wav = waveform.to(device)

    # --- MelSpectrogram en amplitude (power=1.0) ---
    mel_transform = T.MelSpectrogram(
        sample_rate=sr,
        n_fft=n_fft,
        hop_length=hop_length,
        n_mels=n_mels,
        power=1.0,
        center=True,
        pad_mode="reflect",
        norm="slaney",
        mel_scale="slaney",
    ).to(device)

    S_amp = mel_transform(wav)  # (1, n_mels, T)

    # --- Conversion amplitude -> dB (ref = max) ---
    db_transform = T.AmplitudeToDB(stype="amplitude", top_db=None)
    S_DB = db_transform(S_amp)
    S_DB = S_DB - S_DB.max()

    # --- Vers numpy + flip vertical + normalisation ---
    S_DB_np = S_DB.squeeze(0).cpu().numpy()
    S_DB_np = np.flipud(S_DB_np)
    norm = (S_DB_np - S_DB_np.min()) / (S_DB_np.max() - S_DB_np.min() + 1e-8)
    img_arr = (norm * 255).astype(np.uint8)

    # --- Image + resize + sauvegarde ---
    img = Image.fromarray(img_arr, mode="L")
    img = img.resize((512, 256), Image.Resampling.LANCZOS)
    img.save(out_path)


def generate_spectrogrammes(ds: pd.DataFrame, device: str = "cpu"):
    try:
        print("Génération des spectrogrammes Mel (amplitude dB)..", end="")
        l_task = len(ds["filename"])
        for i, (fn, path_in, path_out) in enumerate(
            zip(ds["filename_wav"], ds["path_wav"], ds["path"])
        ):
            print(f"\rProgression : {(100*(i/l_task)):.2f}% | {fn}               ", end="", flush=True)
            mel_spectrogram_amplitude(path_in, path_out, device=device)
        print("\r..[OK]                                                                   ", flush=True)
    except Exception as e:
        print(f"\nErreur lors du téléchargement : {e}.")

# 1. ECAM (Enhanced Channel Attention Mechanism)
C'est un SE-block classique, avec un twist : une mise au carré après le scaling pour accentuer le contraste entre canaux importants/non-importants.

In [4]:
import torch
import torch.nn as nn

class ECAM(nn.Module):
    """
    Enhanced Channel Attention Mechanism
    Squeeze -> Excitation -> Scale -> contrast amplification (square)
    """
    def __init__(self, channels: int, reduction: int = 4):
        super().__init__()
        # Squeeze : global average pooling spatial -> vecteur (B, C, 1, 1)
        self.squeeze = nn.AdaptiveAvgPool2d(1)

        # Excitation : 2 FC layers, le papier dit "1/4 du nb de canaux" pour la 1ère couche
        hidden = max(channels // reduction, 1)
        self.excitation = nn.Sequential(
            nn.Linear(channels, hidden, bias=False),
            nn.ReLU(inplace=True),          # delta dans le papier
            nn.Linear(hidden, channels, bias=False),
            nn.Sigmoid()                     # sigma dans le papier -> poids dans [0,1]
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, c, h, w = x.shape

        # Squeeze: (B,C,H,W) -> (B,C)
        s = self.squeeze(x).view(b, c)

        # Excitation: (B,C) -> (B,C), poids par canal
        weights = self.excitation(s).view(b, c, 1, 1)

        # Scale
        scaled = x * weights

        # Amplification du contraste (spécifique ECAM vs SE classique)
        # le papier insiste sur le "squaring operation to accentuate contrast"
        out = scaled * scaled.sign() * scaled.abs()  # garde le signe, ^2 sur la magnitude
        # version plus simple si tu veux juste suivre le papier littéralement :
        # out = scaled ** 2 * scaled.sign()  -- équivalent
        return out

# 2. Module CNN amélioré (autour de l'ECAM)

In [5]:
class ImprovedCNNBlock(nn.Module):
    """
    Conv1x1 -> ReLU6
    Conv -> ReLU6 -> BatchNorm
    ECAM
    Conv (downscale)
    + residual connection input/output
    """
    def __init__(self, in_channels: int, out_channels: int, stride: int = 1):
        super().__init__()

        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False)
        self.act1 = nn.ReLU6(inplace=True)

        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3,
                                stride=stride, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.act2 = nn.ReLU6(inplace=True)

        self.ecam = ECAM(out_channels)

        # downscale conv (3e couche du papier)
        self.conv3 = nn.Conv2d(out_channels, out_channels, kernel_size=1, bias=False)

        # Connexion résiduelle : projection si in != out ou si stride != 1
        self.need_proj = (in_channels != out_channels) or (stride != 1)
        if self.need_proj:
            self.proj = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        identity = self.proj(x) if self.need_proj else x

        out = self.act1(self.conv1(x))
        out = self.act2(self.bn2(self.conv2(out)))
        out = self.ecam(out)
        out = self.conv3(out)

        return out + identity

# 3. Module ViT amélioré (la partie la plus originale)
Idée : au lieu de "patchify → linear projection → tokens", on fait "patchify → conv locale par patch → unfold en séquence → Transformer → fold → fusion par conv 1×1 avec skip connection".

In [6]:
class LocalConvPatch(nn.Module):
    """
    Découpe en patches, applique une conv locale indépendante par patch
    (= 'Local representations' dans le papier)
    """
    def __init__(self, in_channels: int, embed_dim: int, patch_size: int = 4):
        super().__init__()
        self.patch_size = patch_size
        # conv avec kernel=stride=patch_size -> équivalent à une conv "par patch"
        # mais avec un vrai noyau appris localement plutôt qu'un simple flatten+linear
        self.local_conv = nn.Conv2d(
            in_channels, embed_dim,
            kernel_size=patch_size, stride=patch_size
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, C, H, W) -> (B, embed_dim, H/p, W/p)
        return self.local_conv(x)


class ImprovedViTBlock(nn.Module):
    """
    Local conv patchify -> downsample conv -> unfold en séquence
    -> Transformer encoder (L layers)
    -> fold -> conv 1x1 pour ré-aligner les canaux -> fusion (skip) avec input
    """
    def __init__(self, channels: int, patch_size: int = 4,
                 num_heads: int = 4, depth: int = 2, mlp_ratio: float = 2.0):
        super().__init__()
        self.patch_size = patch_size
        self.channels = channels

        # Local representations
        self.local_patch = LocalConvPatch(channels, channels, patch_size)

        # downsample additionnel (stride=2 conv) mentionné dans le papier
        # pour réduire le nb de points avant le Transformer
        self.downsample = nn.Conv2d(channels, channels, kernel_size=3,
                                     stride=2, padding=1)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=channels,
            nhead=num_heads,
            dim_feedforward=int(channels * mlp_ratio),
            activation="relu",
            batch_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=depth)

        # "deconvolution" / upsample pour revenir à la résolution du downsample
        self.upsample = nn.ConvTranspose2d(channels, channels, kernel_size=3,
                                            stride=2, padding=1, output_padding=1)

        # Conv 1x1 pour réaligner les canaux avant fusion
        self.channel_align = nn.Conv2d(channels, channels, kernel_size=1)

        # Fusion finale (le papier utilise une conv 3x3, "k=3" mentionné en exemple)
        self.fusion_conv = nn.Conv2d(channels * 2, channels, kernel_size=3, padding=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        identity = x  # feature map d'entrée, conservée pour la fusion finale

        # 1. Local representations (conv locale par patch)
        local_feat = self.local_patch(x)          # (B, C, H/p, W/p)

        # 2. Downsample pour réduire le nb de points
        down = self.downsample(local_feat)         # (B, C, H', W')
        b, c, h, w = down.shape

        # 3. Unfold -> séquence de tokens pour le Transformer
        seq = down.flatten(2).transpose(1, 2)      # (B, N, C)  avec N = h*w

        # 4. Transformer (attention globale sur les tokens locaux déjà convolués)
        seq = self.transformer(seq)                # (B, N, C)

        # 5. Fold -> retour en feature map spatiale
        folded = seq.transpose(1, 2).reshape(b, c, h, w)

        # 6. Upsample pour revenir à la résolution de local_feat
        up = self.upsample(folded)
        # sécurité si arrondi de taille (interpolation au besoin)
        if up.shape[-2:] != local_feat.shape[-2:]:
            up = nn.functional.interpolate(up, size=local_feat.shape[-2:],
                                            mode="bilinear", align_corners=False)

        # 7. Réalignement des canaux
        up = self.channel_align(up)

        # 8. Remise à la résolution de l'input original pour la fusion
        if up.shape[-2:] != identity.shape[-2:]:
            up = nn.functional.interpolate(up, size=identity.shape[-2:],
                                            mode="bilinear", align_corners=False)

        # 9. Fusion par concat + conv 3x3 (skip connection avec l'input)
        fused = self.fusion_conv(torch.cat([up, identity], dim=1))
        return fused

In [ ]:
####### FONCTIONS DE DIVERSES #######
#
# Fonction de suppression du dossier local ###
def delete_all_data():
    # Suppression du domaine de data LOCAL
    if os.path.exists(PATH_BASE):
        shutil.rmtree(PATH_BASE)
###

# Fonction de récupératio ndes données sur le S3 ###
def get_from_s3():
    try:
        # Instanciation client boto3
        print("Initialisation du client..")
        prefix = "music-database/"
        s3 = boto3.resource("s3")
        bucket = s3.Bucket(DATA_S3) 
        print("..fait.")
        # Récuparation des datasetq
        print("[Récupération des données depuis {}]".format(DATA_S3))
        print("Téléchargement du dataset en cours..")
        bucket.download_file(prefix + PATH_DS, PATH_DS)
        bucket.download_file(prefix + PATH_DS_SPLIT, PATH_DS_SPLIT)
        print("..fait.")
        # Récupéation sons, images, harmo et percu
        print("Téléchargement des éléments en cours..")
        l_task = [str(k.key) for k in bucket.objects.filter(Prefix = prefix + PATH_SOUND)]
        l_task.extend([str(k.key) for k in bucket.objects.filter(Prefix = prefix + PATH_IMAGE)])
        l_task.extend([str(k.key) for k in bucket.objects.filter(Prefix = prefix + PATH_HARMO)])
        l_task.extend([str(k.key) for k in bucket.objects.filter(Prefix = prefix + PATH_PERCU)])
        # Affichage de la progression
        for s3_key in l_task:
            print(f"\r Progression : {((l_task.index(s3_key)/len(l_task))*100):.2f}% | {s3_key}.       ", end="", flush=True)
            path_out = os.path.dirname(s3_key).replace(prefix, "")
            file_out = s3_key.replace(prefix, "")
            os.makedirs(path_out, exist_ok = True)
            bucket.download_file(s3_key, file_out)
        print("\r Téléchargement des sons....[OK].                            \n", flush = True)
        print("[Données S3 récupérées]")
    # Gestion de l'exception
    except Exception as e:
        print(f"\nErreur lors du téléchargement : {e}.")
        delete_all_data()
        
        
# Fonction de création des répertoires locaux de data ###
def rep_cnn_audio():
    # Si le répertoire de base n'existe pas
    if not os.path.exists(PATH_BASE):
        # Création du répertoire local
        print("Le dossier de base n'existe pas, création lancée.")
        print(f"Création de {PATH_BASE}..", end = "")
        os.makedirs(PATH_BASE)
        print("..[OK]")
        print(f"Création de {PATH_BASE}..", end = "")
        os.makedirs(PATH_DATA)
        print("..[OK]")
        # Récupération depuis le s3
        get_from_s3()
                
        # PATH_SPEC_VIT
        print(f"Création de {PATH_SPEC_VIT}..", end = "")
        os.makedirs(PATH_SPEC_VIT)
        print("..[OK]")
        
        # SOus-répertoires des spectrogrammes par genre
        l_subdirs = [d for d in os.listdir(PATH_SOUND)]
        print(f"Création des {len(l_subdirs)} en cours..", end="")
        for d in os.listdir(PATH_SOUND):
            os.makedirs(PATH_IMAGE + "/" + d)
            os.makedirs(PATH_HARMO + "/" + d)
            os.makedirs(PATH_PERCU + "/" + d)
        print("..[OK]")
        print("Dossier de base créé.")
        print("Lancement de la récupétation S3.")
        get_from_s3()
    else:
        print("Le dossier de base existe, poursuite de l'entrainement.")
###
#

In [ ]:
# Fonction de préparation de dataset ###
def prepare_dataset() -> pd.DataFrame:
    # Exécution de la création si non existant : création des répertoires, sous-répertoires, récupération des fichier audio et génération des spectrogrammes
    rep_cnn_audio()
    # Préparation du dataset, en ignorant jazz.00054.wav (corrompu), puis tri par labels
    ds = pd.read_csv(PATH_DS, encoding = "utf-8")
    ds = ds[ds["filename"] != "jazz.00054.wav"]
    ds = ds.sort_values(by = "label", ascending = True)
    # Suppression des colonnes non-utilisées
    c_to_drop = [c for c in ds.columns if c not in ["filename", "label"]]
    ds = ds.drop(columns = c_to_drop)
    # Modification/Création des colonnes utilisées, chemins des fichiers images
    ds["filename_wav"] = ds["filename"]
    ds["filename"] = [str.replace(c, ".wav", ".png").replace(".0", "0") for c in ds["filename"]]
    ds["path_harmo"] = [PATH_HARMO + "/" + c + "/" + f for c, f in zip(ds["label"], ds["filename"])]
    ds["path_percu"] = [PATH_PERCU + "/" + c + "/" + f for c, f in zip(ds["label"], ds["filename"])]
    #PATH_SPEC_VIT
    ds["path_spec_ViT"] = [PATH_SPEC_VIT + "/" + c + "/" + f for c, f in zip(ds["label"], ds["filename"])]
    
    ds["path"] = [PATH_IMAGE + "/" + c + "/" + f for c, f in zip(ds["label"], ds["filename"])]
    ds["path_wav"] = [PATH_SOUND + "/" + c + "/" + f for c, f in zip(ds["label"], ds["filename_wav"])]
    # Retour du ds
    return ds
###

# 4. Modèle complet

In [10]:
class ImprovedViTModel(nn.Module):
    """
    Stack de blocs [ImprovedViTBlock -> ImprovedCNNBlock] suivi d'une tête de classif.
    Reproduit l'architecture de la Fig.1 du papier (orange = ViT, vert = CNN+ECAM).
    """
    def __init__(self, in_channels: int = 1, num_classes: int = 10,
                 base_channels: int = 64, patch_size: int = 4,
                 num_stages: int = 3, vit_depth: int = 2, num_heads: int = 4):
        super().__init__()

        self.stem = nn.Sequential(
            nn.Conv2d(in_channels, base_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_channels),
            nn.ReLU6(inplace=True),
        )

        stages = []
        ch = base_channels
        for i in range(num_stages):
            stages.append(ImprovedViTBlock(ch, patch_size=patch_size,
                                            num_heads=num_heads, depth=vit_depth))
            next_ch = ch * 2 if i < num_stages - 1 else ch
            stages.append(ImprovedCNNBlock(ch, next_ch, stride=2))
            ch = next_ch
        self.stages = nn.Sequential(*stages)

        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(ch, num_classes)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.stem(x)
        x = self.stages(x)
        return self.head(x)


# Test rapide de cohérence dimensionnelle
if __name__ == "__main__":
    model = ImprovedViTModel(in_channels=1, num_classes=10, patch_size=4)
    dummy = torch.randn(2, 1, 128, 128)  # ex: Mel spectrogramme 128x128
    out = model(dummy)
    print(out.shape)  # torch.Size([2, 10])

torch.Size([2, 10])


In [15]:
import os
import sys
import argparse
import datetime
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import mlflow
import mlflow.pytorch
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from mlflow.models.signature import infer_signature
from torchinfo import summary
from torchviz import make_dot
from PIL import Image
from torchvision import transforms

# ---- Imports locaux (tes modules) ----
# from models import ImprovedViTModel          # architecture définie dans nos échanges
# from preprocessing import generate_spectrogrammes  # fonction torchaudio locale

# MLFLOW_URI = "http://localhost:5000"
# PATH_BASE   = "./data/spectrograms"           # répertoire de sortie des spectrogrammes


# ===========================================================================
# DATASET
# ===========================================================================
def trainval_mytransform():
    return transforms.Compose([
        transforms.Resize((128, 128)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(5),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5], std=[0.5]),
    ])

def test_mytransform():
    return transforms.Compose([
        transforms.Resize((128, 128)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5], std=[0.5]),
    ])


class ImageDataset(Dataset):
    """
    Charge les spectrogrammes Mel (images .png) depuis le disque.
    Compatible avec le pipeline generate_spectrogrammes / mel_spectrogram_amplitude.
    """
    def __init__(self, df: pd.DataFrame, mytransforms=None, mymapping: dict = None):
        self.df = df.reset_index(drop=True)
        self.transforms = mytransforms
        self.mapping = mymapping  # {"blues": 0, "classical": 1, ...}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["path"]).convert("L")   # "L" = grayscale, 1 canal
        if self.transforms:
            img = self.transforms(img)
        label = self.mapping[row["label"]]
        return img, label


# ===========================================================================
# EARLY STOPPING
# ===========================================================================
class EarlyStopping:
    def __init__(self, patience: int = 7, mode: str = "min", filepath: str = "best_model.pth"):
        self.patience = patience
        self.mode = mode
        self.filepath = filepath
        self.best = float("inf") if mode == "min" else -float("inf")
        self.counter = 0
        self.stopped = False

    def step(self, metric: float, model: nn.Module) -> bool:
        improved = (self.mode == "min" and metric < self.best) or \
                   (self.mode == "max" and metric > self.best)
        if improved:
            self.best = metric
            self.counter = 0
            torch.save(model.state_dict(), self.filepath)
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stopped = True
        return self.stopped


# ===========================================================================
# TRAINING LOOP
# ===========================================================================
def train_process(model, train_loader, val_loader, criterion, optimizer,
                  epochs, patience, filepath, early_stop_mode, device):
    model.to(device)
    es = EarlyStopping(patience=patience, mode=early_stop_mode, filepath=filepath)

    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

    for epoch in range(1, epochs + 1):
        # --- Train ---
        model.train()
        train_loss, train_correct, train_total = 0.0, 0, 0
        for X, y in train_loader:
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()
            out = model(X)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * X.size(0)
            train_correct += (out.argmax(1) == y).sum().item()
            train_total += X.size(0)

        # --- Validation ---
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for X, y in val_loader:
                X, y = X.to(device), y.to(device)
                out = model(X)
                loss = criterion(out, y)
                val_loss += loss.item() * X.size(0)
                val_correct += (out.argmax(1) == y).sum().item()
                val_total += X.size(0)

        # --- Métriques epoch ---
        tl = train_loss / train_total
        ta = train_correct / train_total
        vl = val_loss / val_total
        va = val_correct / val_total
        diff_loss = val_loss - train_loss

        history["loss"].append(tl)
        history["val_loss"].append(vl)
        history["accuracy"].append(ta)
        history["val_accuracy"].append(va)
        history["diff_loss"].append(diff_loss)
        

        print(f"Epoch {epoch:03d}/{epochs} | "
              f"train_loss={tl:.4f} acc={ta:.4f} | "
              f"val_loss={vl:.4f} acc={va:.4f} | "
              f"ES counter={es.counter}/{patience}")

        # MLflow log par epoch
        mlflow.log_metrics({
            "train_loss": tl, "train_acc": ta,
            "val_loss": vl,   "val_acc": va,
            "loss_diff":diff_loss
        }, step=epoch)

        if es.step(vl, model):
            print(f"Early stopping déclenché à l'epoch {epoch}.")
            break

    return history


# ===========================================================================
# LOG HISTORIQUE MLFLOW (courbes loss/acc)
# ===========================================================================
def mlflow_log_historic(history: dict):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history["train_loss"], label="train")
    axes[0].plot(history["val_loss"],   label="val")
    axes[0].set_title("Loss")
    axes[0].legend()
    axes[1].plot(history["train_acc"], label="train")
    axes[1].plot(history["val_acc"],   label="val")
    axes[1].set_title("Accuracy")
    axes[1].legend()
    plt.tight_layout()
    mlflow.log_figure(fig, "train_val_curves.png")
    plt.close(fig)


# ===========================================================================
# MAIN
# ===========================================================================
if __name__ == "__main__":

    parser = argparse.ArgumentParser()
    parser.add_argument("--batch_size",         type=int,   default=32)
    parser.add_argument("--lr",                 type=float, default=1e-4)
    parser.add_argument("--weight_decay",       type=float, default=0.01)
    parser.add_argument("--random_state",       type=int,   default=42)
    parser.add_argument("--test_size",          type=float, default=0.2)
    parser.add_argument("--val_size",           type=float, default=0.2)
    parser.add_argument("--experiment_name",    type=str,   default="improved_vit_audio")
    parser.add_argument("--model_name", type=str, default="audio_classifier")
    parser.add_argument("--model_version",      type=str,   default="v1",
                        choices=["v1"],         help="v1=ImprovedViTModel")
    parser.add_argument("--optimizer_name",     type=str,   default="AdamW",
                        choices=["Adam", "AdamW"])
    parser.add_argument("--n_epochs",           type=int,   default=200)
    parser.add_argument("--filepath",           type=str,   default="best_model_vit.pth")
    parser.add_argument("--mlflow_model_name",  type=str,   default="improved_vit_audio_classifier")
    parser.add_argument("--patience",           type=int,   default=15)
    parser.add_argument("--early_stop",         type=str,   default="min",
                        choices=["min", "max"])
    # Hyperparamètres ViT (spécifiques à ImprovedViTModel)
    parser.add_argument("--patch_size",         type=int,   default=4)
    parser.add_argument("--base_channels",      type=int,   default=64)
    parser.add_argument("--num_stages",         type=int,   default=3)
    parser.add_argument("--vit_depth",          type=int,   default=2)
    parser.add_argument("--num_heads",          type=int,   default=4)
    args = parser.parse_args()

    NUM_CLASSES = 10
    # Récupération des arguments
    BATCH_SIZE = args.batch_size
    TEST_SIZE = args.test_size
    VAL_SIZE = args.val_size
    RANDOM_STATE = args.random_state
    LR = args.lr
    WEIGHT_DECAY = args.weight_decay
    NUM_CLASSES = 10
    EXPERIMENT_NAME = args.experiment_name
    MODEL_NAME = args.model_name
    MODEL_VERSION = args.model_version
    OPTIMIZER_NAME = args.optimizer_name
    N_EPOCHS = args.n_epochs
    FILEPATH = args.filepath
    PATIENCE = args.patience
    MLFLOW_MODEL_NAME = args.mlflow_model_name
    EARLY_STOP=args.early_stop
    
    # --- Device ---
    DEVICE = torch.accelerator.current_accelerator().type \
             if torch.accelerator.is_available() else "cpu"
    print(f"Device : {DEVICE}")
    if DEVICE == "cpu":
        print("Attention : entraînement sur CPU, ce sera lent.")
        # Pas de sys.exit() ici contrairement à l'original :
        # ImprovedViTModel reste utilisable en CPU pour du debug/test

    # --- Dataset ---
    ds = prepare_dataset()   # ta fonction existante -> DataFrame avec colonnes label/path/path_wav/etc.
    unique_labels = sorted(ds["label"].unique().tolist())
    mapping_LI = {l: i for i, l in enumerate(unique_labels)}
    reverse_LI = {v: k for k, v in mapping_LI.items()}
    print(f"Mapping : {mapping_LI}")

    # --- Splits ---
    ds_train, ds_test = train_test_split(
        ds, test_size=args.test_size, random_state=args.random_state, stratify=ds["label"])
    ds_train, ds_val = train_test_split(
        ds_train, test_size=args.val_size, random_state=args.random_state, stratify=ds_train["label"])
    print(f"Train={ds_train.shape} | Val={ds_val.shape} | Test={ds_test.shape}")

    # --- Génération des spectrogrammes si absents ---
    if not os.path.exists(PATH_BASE):
        generate_spectrogrammes(ds, device=DEVICE)

    # --- DataLoaders ---
    ids_train = ImageDataset(ds_train, mytransforms=trainval_mytransform(), mymapping=mapping_LI)
    ids_val   = ImageDataset(ds_val,   mytransforms=test_mytransform(),     mymapping=mapping_LI)
    ids_test  = ImageDataset(ds_test,  mytransforms=test_mytransform(),     mymapping=mapping_LI)

    train_loader = DataLoader(ids_train, batch_size=args.batch_size, shuffle=True,  drop_last=True)
    val_loader   = DataLoader(ids_val,   batch_size=args.batch_size, shuffle=False)
    test_loader  = DataLoader(ids_test,  batch_size=args.batch_size, shuffle=False)

    # --- Modèle ---
    # in_channels=1 car spectrogramme grayscale (mode L)
    model = ImprovedViTModel(
        in_channels=1,
        num_classes=NUM_CLASSES,
        patch_size=args.patch_size,
        base_channels=args.base_channels,
        num_stages=args.num_stages,
        vit_depth=args.vit_depth,
        num_heads=args.num_heads,
    )

    criterion = nn.CrossEntropyLoss()
    optimizers = {
        "Adam":  optim.Adam(model.parameters(),  lr=args.lr, weight_decay=args.weight_decay),
        "AdamW": optim.AdamW(model.parameters(), lr=args.lr, weight_decay=args.weight_decay),
    }
    optimizer = optimizers[args.optimizer_name]

    soft_dt = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

    # --- MLflow ---
    mlflow.set_tracking_uri(MLFLOW_URI)
    mlflow.set_experiment(args.experiment_name)

    with mlflow.start_run(run_name=f"ImprovedViT_{soft_dt}"):

        mlflow.log_params({
            "batch_size":    args.batch_size,
            "learning_rate": args.lr,
            "weight_decay":  args.weight_decay,
            "epochs":        args.n_epochs,
            "patience":      args.patience,
            "num_classes":   NUM_CLASSES,
            "device":        DEVICE,
            "patch_size":    args.patch_size,
            "base_channels": args.base_channels,
            "num_stages":    args.num_stages,
            "vit_depth":     args.vit_depth,
            "num_heads":     args.num_heads,
            "optimizer":     args.optimizer_name,
        })

        # --- Entraînement ---
        history = train_process(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            criterion=criterion,
            optimizer=optimizer,
            epochs=args.n_epochs,
            patience=args.patience,
            filepath=args.filepath,
            early_stop_mode=args.early_stop,
            device=DEVICE,
        )

        # --- Evaluation sur le test set ---
        best_model = ImprovedViTModel(
            in_channels=1,
            num_classes=NUM_CLASSES,
            patch_size=args.patch_size,
            base_channels=args.base_channels,
            num_stages=args.num_stages,
            vit_depth=args.vit_depth,
            num_heads=args.num_heads,
        )
        best_model.load_state_dict(torch.load(args.filepath, map_location=DEVICE))
        best_model.to(DEVICE)
        best_model.eval()

        Y_true, Y_pred = [], []
        with torch.no_grad():
            for X, y in test_loader:
                X, y = X.to(DEVICE), y.to(DEVICE)
                out = best_model(X)
                Y_pred.extend(out.argmax(1).cpu().numpy())
                Y_true.extend(y.cpu().numpy())

        Y_true = np.array(Y_true)
        Y_pred = np.array(Y_pred)

        test_acc = accuracy_score(Y_true, Y_pred)
        test_f1  = f1_score(Y_true, Y_pred, average="macro")
        print(f"\nTest accuracy={test_acc:.4f} | F1-macro={test_f1:.4f}")

        mlflow.log_metrics({"TEST_accuracy": test_acc, "TEST_f1_macro": test_f1})

        # --- Log modèle MLflow ---
        dummy_input = torch.randn(1, 1, 128, 128).to(DEVICE)   # (B, C, H, W) — grayscale 128x128
        with torch.no_grad():
            dummy_output = best_model(dummy_input)
        signature = infer_signature(
            dummy_input.cpu().numpy(),
            dummy_output.cpu().numpy()
        )
        mlflow.pytorch.log_model(
            pytorch_model=best_model,
            name=args.mlflow_model_name,
            registered_model_name=args.mlflow_model_name,
            signature=signature,
            input_example=dummy_input.cpu().numpy(),
        )

        # --- Matrice de confusion ---
        class_names = [reverse_LI[i] for i in range(NUM_CLASSES)]
        fig, ax = plt.subplots(figsize=(10, 8))
        ConfusionMatrixDisplay.from_predictions(
            Y_true, Y_pred,
            display_labels=class_names,
            normalize="true",
            cmap="Blues",
            xticks_rotation=45,
            values_format=".0%",
            ax=ax,
        )
        plt.title("Matrice de confusion — ImprovedViT")
        mlflow.log_figure(fig, "confusion_matrix_vit.png")
        plt.close(fig)

        # --- Graphe du modèle (torchviz) ---
        y_dot = best_model(torch.randn(1, 1, 128, 128).to(DEVICE))
        dot = make_dot(y_dot, params=dict(best_model.named_parameters()))
        img_path = dot.render("model_graph_vit", format="png", cleanup=True)
        mlflow.log_artifact(img_path)

        # --- Summary torchinfo ---
        model_summary = summary(best_model, input_size=(1, 1, 128, 128), verbose=0)
        with open("model_summary_vit.txt", "w") as f:
            f.write(str(model_summary))
        mlflow.log_artifact("model_summary_vit.txt")

        # --- Courbes loss/acc ---
        mlflow_log_historic(history)

    print("Run MLflow terminé.")

Device : cpu
Attention : entraînement sur CPU, ce sera lent.
Le dossier de base n'existe pas, création lancée.
Création de gtzan-dataset-music-genre-classification....[OK]
Création de gtzan-dataset-music-genre-classification....[OK]
Initialisation du client..

Erreur lors du téléchargement : Missing Dependency: Using the login credential provider requires an additional dependency. You will need to pip install "botocore[crt]" before proceeding..
Création de gtzan-dataset-music-genre-classification/Data/img_ViT....[OK]


FileNotFoundError: [WinError 3] Le chemin d’accès spécifié est introuvable: 'gtzan-dataset-music-genre-classification/Data/genres_original'